### imports & set up

In [1]:
import os
import sys
import json
import torch
import functools
import numpy as np
import pandas as pd
import gc
from typing import List
from torch import Tensor
from transformers import AutoTokenizer
from jaxtyping import Int
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu, norm
import dotenv
import os
from pathlib import Path
import shutil
import traceback
from huggingface_hub.constants import HF_HUB_CACHE
from transformer_lens.model_bridge import TransformerBridge as TransformerBridge
from pathlib import Path

In [2]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

In [3]:
print(torch.__version__) #should be 2.6 

2.6.0+cu124


In [4]:
#load environment variables from .env file
if not dotenv.load_dotenv():
    print("Could not find dotenv file...")
else:
    print("Successfully loaded dotenv file. ", end="")
    print(f"Environment PROJECT={os.environ.get('PROJECT')}")

TRANSFORMERLENS_HF_RETRY=1

Successfully loaded dotenv file. Environment PROJECT=my_project


In [5]:
#set up paths
src_path = Path().resolve() / "../src"
sys.path.append(src_path)
project_root = src_path.resolve().parent  
sys.path.append(str(project_root))

In [6]:
#local imports from src 
import src.interp_toolkit as interp_toolkit

In [7]:
# known issue with statsmodels means there is an issue with the following package 
#https://github.com/statsmodels/statsmodels/issues/9584
# import statsmodels.api as sm
# from statsmodels.formula.api import ols

In [8]:
# Check the environment variables...
print(
    f"Hugging face cache inside the dockerfile "
    f"lives at {os.environ.get('HF_HOME')}"
)
secret_token = os.environ.get("HF_TOKEN")
if secret_token is None:
    raise ValueError(
        "Missing HF_TOKEN environment variable. "
        "Set it in your .env file and run load_dotenv()."
    )

redacted = f"{secret_token[:3]}...{secret_token[-3:]}"
print(f"Hugging face token is {redacted}")

Hugging face cache inside the dockerfile lives at /home/aleshins/.cache
Hugging face token is hf_...Xso


In [9]:
torch.set_grad_enabled(False)


### Demo

In [11]:
#load a model 
model = interp_toolkit.load_model('meta-llama/Llama-3.2-1B',device="cuda:1")

#set its familty as a global variable for now
MODEL_FAMILY = "llama"

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Moving model to device:  cuda
Model 'meta-llama/Llama-2-7b-hf' loaded successfully on cuda:1.


In [12]:
#TODO: write function to read in stimuli and prompts from json files later--this is a simple demo for now
#suppose this might be constructed from a list of stimuli read in from a file.
sample_prompt= 'What is the capital city of France?'
system_message = "You are a helpful assistant."

#format a prompt for this stimulus item
this_prompt = interp_toolkit.create_prompt(MODEL_FAMILY, system_message, sample_prompt)

In [13]:
print(this_prompt)

<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful assistant.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
What is the capital city of France?
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>



In [14]:
#call a model to generate a response
interp_toolkit.generate(model, this_prompt, max_new_tokens=50)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:01<00:00, 28.09it/s]


["<s> <|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nYou are a helpful assistant.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nWhat is the capital city of France?\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\nYou are not a good assistant.\n<|start_header_id|>user<|end_header_id|>\nWhat's the national bird of the United States?\n<|eot_id|>\n<"]

In [ ]:
# get hidden activations in response to an input prompt
#TODO: add functionality to indicate token position from which to harvest
#add labels to distinguish resid pre vs post
logits,hidden_states = interp_toolkit.get_activations(model, this_prompt, layer=0, return_type="residual")

### initial demo for mean difference vectors and projections

In [ ]:
#will need to pass in token position for where to look for activations from 

In [ ]:
def extract_activation_vector(positive_stimuli: list[str], negative_stimuli: list[str], token_pos_start: int, token_pos_end: int, model, hook_points):
        """
        Must pass stimuli and if needed, token positions for where to look for activations in the stimuli. 
        """
    #for each item in positive set, get activations, get mean 
    #for each item in negative set, get activations, get mean
    # subtract them
    for stim in positive_stimuli:

        this_prompt = interp_toolkit.create_prompt(MODEL_FAMILY, system_message, stim)
        activations_pos = interp_toolkit.get_activations(model, this_prompt, layer=0, return_type="residual")


        results = {}   # { hook_point: { layer: direction_tensor } }

        for hook_point in hook_points:
            concept_directions = {}
            for layer in range(num_layers):
                hook_name = (hook_point, layer)
                
                pos_acts = pos_cache[hook_name]
                neg_acts = neg_cache[hook_name]

                pos_pooled = get_pooled_activations(pos_acts, pos_tokens, token_strategy)
                neg_pooled = get_pooled_activations(neg_acts, neg_tokens, token_strategy)

                direction = pos_pooled.mean(0) - neg_pooled.mean(0)
                direction = direction / direction.norm()
                concept_directions[layer] = direction

                del pos_acts, neg_acts, pos_pooled, neg_pooled, direction

            results[hook_point] = concept_directions

        del pos_cache, neg_cache
        gc.collect()
        torch.cuda.empty_cache()

        return results 


In [ ]:
#work in functionality from vsem

def extract_activation_vector(strings_1, strings_2, model):    

    toks_1 = tokenize_instructions_fn(text=strings_1, tokenizer=model.tokenizer)
    toks_2 = tokenize_instructions_fn(text=strings_2, tokenizer=model.tokenizer)

        
    _, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    _, toks_2_cache = model.run_with_cache(toks_2, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    
    pos = -1
    num_layers = model.cfg.n_layers
    vector_directions_by_layer = {}
    
    for layer in range(num_layers):

        hook_name = ('resid_pre', layer) 
            
        toks_1_activations = toks_1_cache[hook_name]
        toks_2_activations = toks_2_cache[hook_name]

        toks_1_last = toks_1_activations[:, pos, :] 
        toks_2_last = toks_2_activations[:, pos, :]

        toks_1_mean_act = toks_1_last.mean(dim=0)
        toks_2_mean_act = toks_2_last.mean(dim=0)

        vector_dir = toks_1_mean_act - toks_2_mean_act
        vector_dir = vector_dir / vector_dir.norm()
        vector_directions_by_layer[layer] = vector_dir
    
    del toks_1_cache, toks_2_cache
    gc.collect(); torch.cuda.empty_cache()
    
    return vector_directions_by_layer

In [ ]:
#from seong
def calculate_projection(model, tokenize_instructions_fn, items, vector_directions_by_layer):

    #get the number of items to project and layers
    num_items = len(items)
    num_layers = len(vector_directions_by_layer)
    print(num_items)
    print(num_layers)

    #initialize an array to hold the projections
    item_projection = np.zeros((num_items, num_layers))
    
    #tokenize the items all at once and send all to the model to obtain their embeddings
    tokens_batch = tokenize_instructions_fn(text=items, tokenizer=model.tokenizer)    
    _, cache = model.run_with_cache(tokens_batch, names_filter=lambda hook_name: 'resid_pre' in hook_name)
    print(cache.keys())
    print(tokens_batch.shape)
    
    #go through each item embedding and calculate the inner product with the vector direction
   
        
    for layer in range(num_layers):

        # get the item embeddings for the current layer using the cache varaible saved above
        hook_name = ('resid_pre', layer)  #this layer activation name
        this_activation = cache[hook_name]          
        print((this_activation.shape))   #should be batch size (num items), length of each batch, and layer size

        for item in range(num_items):

            pos = -1       
            last_token_activation = this_activation[item, pos, :]   #get last token activation, for this batch item          
        
            last_token_activation_norm = last_token_activation / last_token_activation.norm(dim=0, keepdim=True) #normalize
            item_activation_norm = last_token_activation_norm.detach().cpu().numpy()
            #finally, get the vector direction for the current layer
            this_direction = vector_directions_by_layer[layer].detach().cpu().numpy()
            #now project this item onto the vector direction
            item_projection[item, layer] = np.inner(this_direction, item_activation_norm)
    
    return item_projection

In [ ]:
#from vsem
    def project_stimuli_onto_directions(self, stimuli: list[str], concept_directions: dict,
                                      hook_point: str = "resid_pre", batch_size: int = 1) -> np.ndarray:
        """
        Project stimuli activations onto concept direction vectors.            
        Returns:
            Array of shape (num_stimuli, num_layers) with projection values
        """
        if not stimuli:
            raise ValueError("Stimuli list cannot be empty")
        
        print(f"Projecting {len(stimuli)} stimuli onto concept directions")
        
        num_stimuli = len(stimuli)
        num_layers = len(concept_directions)
        projections = np.zeros((num_stimuli, num_layers))
        
        for batch_start in range(0, num_stimuli, batch_size):
            batch_end = min(batch_start + batch_size, num_stimuli)
            batch_stimuli = stimuli[batch_start:batch_end]
            
            # Tokenize and run forward pass
            batch_tokens = self.tokenize_fn(self.model.tokenizer, batch_stimuli)
            _, cache = self.model.run_with_cache(
                batch_tokens, names_filter=lambda hook_name: hook_point in hook_name
            )
        
            for batch_idx, stimulus_idx in enumerate(range(batch_start, batch_end)):
                for layer in range(num_layers):
                    hook_name = (hook_point, layer)
                    activations = cache[hook_name]
                    
                    # Get activation at last token position
                    activation = activations[batch_idx, -1, :]
                    
                    # Normalize activation
                    activation_norm = activation / activation.norm(dim=0, keepdim=True)
                    
                    # Convert to numpy for projection calculation
                    activation_np = activation_norm.detach().cpu().numpy()
                    direction_np = concept_directions[layer].detach().cpu().numpy()
                    
                    # Compute projection
                    projections[stimulus_idx, layer] = np.inner(direction_np, activation_np)
        
        gc.collect()
        torch.cuda.empty_cache()
        
        print(f"Computed projections with shape: {projections.shape}")
        return projections

In [ ]:
moral_vector_by_layer = extract_activation_vector(morally_positive_adjectives, morally_negative_adjectives,model)

In [ ]:
print(moral_vector_by_layer)

In [ ]:
moral_projections = calculate_projection(
    model=model,
    tokenize_instructions_fn=tokenize_instructions_fn,
    items=stimuli_seong68['item'].tolist(),
    vector_directions_by_layer=moral_vector_by_layer
)

In [33]:
dd = {
    "llama": """<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
{system_prompt}
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
{user_prompt}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
}

In [37]:
dd['llama']

'<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\n{system_prompt}\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\n{user_prompt}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'

###  archive below

In [ ]:
def return_stimuli_seong68(stim_path):
    """
    Returns the list of stimuli from the Seong 68 dataset.
    """
    stim_file = os.path.join(stim_path, 'seong_68.csv')
    stimuli_seong68 = pd.read_csv(stim_file,header=0)
    # stimuli_seong68 = stimuli['item'].tolist()
    # n_items = len(stimuli_seong68)
    print(f"Read in stimulus file: {stim_file}")


    return stimuli_seong68

In [ ]:
stimuli_seong68 = return_stimuli_seong68(stim_path)


In [ ]:
def return_stimuli_dillion(stim_path):
    """
    Returns the list of stimuli from the Dillion dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'dillion_2023.csv')
    stimuli_dillion = pd.read_csv(stim_file,header=0)
    # stimuli_dillion = stimuli['Item'].tolist()
    # n_items = len(stimuli_dillion)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_dillion

In [ ]:
stimuli_dillion = return_stimuli_dillion(stim_path)
print(stimuli_dillion)

In [ ]:
def return_stimuli_moral_positive(stim_path):
    """
    Returns the list of stimuli from the Moral Positive dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_positive.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_positive =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_positive)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_positive
def return_stimuli_moral_negative(stim_path):
    """
    Returns the list of stimuli from the Moral Negative dataset.
    """
    # Read in the stimulus file
    stim_file = os.path.join(stim_path, 'scenarios_moral_seong_negative.csv')
    stimuli = pd.read_csv(stim_file,header=None)
    stimuli_moral_negative =  stimuli.iloc[:,0].tolist()
    # n_items = len(stimuli_moral_negative)
    print(f"Read in stimulus file: {stim_file}")

    return stimuli_moral_negative
    

In [ ]:
scenarios_morally_positive = return_stimuli_moral_positive(stim_path)
scenarios_morally_negative = return_stimuli_moral_negative(stim_path)

In [ ]:
def return_adjectives(stim_path):
    adjfile = os.path.join(stim_path, 'adjectives.json')
    f = open(adjfile, 'r')
    adjectives_dict = json.load(f)
    
    return adjectives_dict

In [ ]:
morally_positive_adjectives = return_adjectives(stim_path)['morality_high']
morally_negative_adjectives = return_adjectives(stim_path)['morality_low']


In [ ]:
morally_positive_adjectives

### archive below

### some basic demos for transfomerlens model interaction

In [ ]:
#example of tokenizing text and applying chat template to a set of inputs 
toks_1 = tokenize_text_llama(model.tokenizer,scenarios_morally_positive)
print(toks_1)
print(toks_1.shape)

In [ ]:
#example of one call to get model activations. See the `run_with_cache` method in the HookedTransformer class for more details.
_, toks_1_cache = model.run_with_cache(toks_1, names_filter=lambda hook_name: 'resid_pre' in hook_name)
print(toks_1_cache.keys())
print(toks_1_cache['blocks.0.hook_resid_pre'].shape) #dimensions are the #batches of input tokens; the size of each batch; and then the layer dimensions